<a href="https://colab.research.google.com/github/riya-rajbhut/music-ai/blob/main/basic-neural-network/rnn_basic_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!sudo apt install -y fluidsynth

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fluidsynth is already the newest version (2.2.5-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [46]:
!pip install --upgrade pyfluidsynth

In [47]:
!pip install pretty_midi

In [5]:
import collections
import datetime
import fluidsynth
import glob
import numpy as np
import pathlib
import pandas as pd
import pretty_midi
import seaborn as sns
import tensorflow as tf

from IPython import display
from matplotlib import pyplot as plt
from typing import Optional

In [6]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

# Sampling rate for audio playback
_SAMPLING_RATE = 16000

In [20]:
import pathlib
import tensorflow as tf

# 1. Download and extract the file, capturing the returned path to the zip file
zip_path = tf.keras.utils.get_file(
    'maestro-v3.0.0-midi.zip',
    origin='https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip',
    extract=True,
    cache_dir='.', cache_subdir='data',
)

# 2. Point to the actual extracted folder inside 'data/'
# Keras extracts the archive contents directly into 'data/', which contains the 'maestro-v3.0.0' folder




In [35]:
#!dir /s data
#!ls -R data/
import pathlib

# 1. Update this to the actual folder path shown in your file listing
data_dir = pathlib.Path('data/maestro-v2_extracted/maestro-v3.0.0')

# 2. Fix the glob pattern to look recursively (**/) through the subfolders
filenames = list(data_dir.glob('**/*.mid*'))

print('Number of files found:', len(filenames))

Number of files found: 1276


# Process a MIDI file
First, use pretty_midi to parse a single MIDI file and inspect the format of the notes.

In [36]:
sample_file = filenames[1]
print(sample_file)

data/maestro-v2_extracted/maestro-v3.0.0/2018/MIDI-Unprocessed_Schubert7-9_MID--AUDIO_11_R2_2018_wav.midi


In [37]:
#Generate a PrettyMIDI object for the sample MIDI file.
pm = pretty_midi.PrettyMIDI(sample_file)

# Why is it hanging?
By default, pm.fluidsynth() synthesizes the entire MIDI file before your code extracts the 30-second snippet (waveform[:seconds*_SAMPLING_RATE]). If your MIDI file is long (e.g., a 10-minute classical piece), generating CD-quality audio at 44.1 kHz takes a massive amount of processing power and memory, forcing your notebook kernel to grind to a halt.

# The Fix:
Trim the MIDI before synthesizing it
Instead of synthesizing the whole file and cutting the audio down, cut the MIDI file to 30 seconds first. This way, FluidSynth only has to process 30 seconds of data, which takes less than a second to execute.

In [ ]:
def display_audio(pm: pretty_midi.PrettyMIDI, seconds=30):
  waveform = pm.fluidsynth(fs=_SAMPLING_RATE)
  # Take a sample of the generated waveform to mitigate kernel resets
  waveform_short = waveform[:seconds*_SAMPLING_RATE]
  # Force Jupyter to display the returned audio player
  return display(display_audio(pm, seconds=30))

#display_audio(pm, seconds=30) #this func is hanging coz its synthecising entire file first (which is taking time) and then cutting for first 30 secs

In [50]:
import copy
import pretty_midi
from IPython import display

def display_audio(pm: pretty_midi.PrettyMIDI, seconds=30):
    # 1. Deep copy the MIDI object so we don't alter the original file
    pm_short = copy.deepcopy(pm)

    # 2. Trim the MIDI data down to the requested duration
    # This prevents fluidsynth from rendering minutes of silent or heavy audio
    pm_short.adjust_times([0, seconds], [0, seconds])

    # 3. Cut off any notes trailing past the limit
    for instrument in pm_short.instruments:
        instrument.notes = [n for n in instrument.notes if n.start < seconds]
        for n in instrument.notes:
            if n.end > seconds:
                n.end = seconds

    # 4. Synthesize only the 30-second chunk
    waveform_short = pm_short.fluidsynth(fs=_SAMPLING_RATE)

    return display.Audio(waveform_short, rate=_SAMPLING_RATE)

# Call and explicitly display the player
# Change 'display(...)' to 'display.display(...)'
display.display(display_audio(pm, seconds=30))